# PHASE 5: Modeling and Validation — NASA RUL

**Reference-aligned objective**
- Reproduce model benchmarking, tuning, and validation used in Kaggle-style NASA RUL pipeline.

**Inputs**
- `../data/processed/train_featured.csv`
- `../data/processed/valid_featured.csv`

**Data-source adaptation notes**
1. Models train on engineered features generated from your cleaned inputs.
2. Feature space is inferred dynamically to respect removed sensors.
3. Group-aware CV uses `unit_number` to avoid leakage across engine trajectories.
4. If selected-feature manifest exists, experiments run on both full and selected feature spaces.

**Validation checkpoints included**
- Fold-level CV metrics
- Holdout validation metrics (RMSE, MAE, R2, NASA score)
- Prediction range and residual diagnostics

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import xgboost as xgb

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120})

DATA_DIR = Path("../data/processed")
TRAIN_PATH = DATA_DIR / "train_featured.csv"
VALID_PATH = DATA_DIR / "valid_featured.csv"
MANIFEST_PATH = DATA_DIR / "feature_manifest.csv"

TARGET = "RUL_clipped"
ID_COL = "unit_number"
TIME_COL = "time_cycles"


def load_featured(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    df = pd.read_csv(path)
    required = [ID_COL, TIME_COL, TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")
    return df


df_train = load_featured(TRAIN_PATH, "train_featured")
df_valid = load_featured(VALID_PATH, "valid_featured")

exclude_cols = {ID_COL, TIME_COL, "RUL", TARGET}
full_feature_cols = [c for c in df_train.columns if c not in exclude_cols]

selected_union = None
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    if {"feature", "selected_union"}.issubset(manifest.columns):
        selected_union = manifest.loc[manifest["selected_union"] == True, "feature"].tolist()
        selected_union = [c for c in selected_union if c in full_feature_cols]

print(f"Train shape: {df_train.shape}")
print(f"Valid shape: {df_valid.shape}")
print(f"Full features: {len(full_feature_cols)}")
print(f"Selected features available: {0 if selected_union is None else len(selected_union)}")

## 1) Evaluation metrics (RMSE + NASA asymmetric score)

NASA score penalizes late prediction errors more than early ones, as in standard turbofan RUL evaluation.

In [ ]:
def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


nasa_scorer = make_scorer(nasa_score, greater_is_better=False)
rmse_scorer = make_scorer(rmse, greater_is_better=False)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

print("✅ Metrics ready")

## 2) Group-aware cross-validation setup

Samples from the same engine stay in the same fold to avoid leakage.

In [ ]:
def get_xy_groups(df: pd.DataFrame, feature_cols: list[str]):
    X = df[feature_cols]
    y = df[TARGET]
    groups = df[ID_COL]
    return X, y, groups


feature_set_map = {"full": full_feature_cols}
if selected_union is not None and len(selected_union) > 10:
    feature_set_map["selected_union"] = selected_union

gkf = GroupKFold(n_splits=5)
print(f"Feature spaces to evaluate: {list(feature_set_map.keys())}")
print("✅ GroupKFold configured")

## 3) Model benchmarking with cross-validation

Benchmark baseline linear, tree ensemble, and gradient boosting (including XGBoost) using identical fold strategy.

In [ ]:
base_models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=350,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    ),
}

cv_rows = []
for feature_name, cols in feature_set_map.items():
    X, y, groups = get_xy_groups(df_train, cols)

    for model_name, model in base_models.items():
        scores = cross_validate(
            model,
            X,
            y,
            groups=groups,
            cv=gkf,
            scoring={"rmse": rmse_scorer, "mae": mae_scorer, "nasa": nasa_scorer},
            n_jobs=-1,
            return_train_score=False,
        )

        cv_rows.append(
            {
                "feature_space": feature_name,
                "model": model_name,
                "cv_rmse": -scores["test_rmse"].mean(),
                "cv_mae": -scores["test_mae"].mean(),
                "cv_nasa": -scores["test_nasa"].mean(),
            }
        )

cv_results = pd.DataFrame(cv_rows).sort_values(["feature_space", "cv_rmse", "cv_nasa"])
cv_results

## 4) Hyperparameter tuning (best CV candidate)

Tune the best candidate model with group-aware CV and NASA-focused secondary objective.

In [ ]:
best_row = cv_results.iloc[0]
best_feature_space = best_row["feature_space"]
best_model_name = best_row["model"]

print("Best CV baseline candidate:")
print(best_row)

feature_cols_for_final = feature_set_map[best_feature_space]
X_train, y_train, groups_train = get_xy_groups(df_train, feature_cols_for_final)

if best_model_name == "XGBoost":
    tune_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1)
    param_dist = {
        "n_estimators": [200, 300, 400, 500],
        "max_depth": [3, 4, 5, 6, 7],
        "learning_rate": [0.01, 0.03, 0.05, 0.08],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
    }
else:
    tune_model = RandomForestRegressor(random_state=42, n_jobs=-1)
    param_dist = {
        "n_estimators": [200, 300, 400],
        "max_depth": [8, 10, 12, 15],
        "min_samples_split": [2, 4, 6],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None],
    }

search = RandomizedSearchCV(
    estimator=tune_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train, groups=groups_train)

best_model = search.best_estimator_
print("Best tuned params:")
print(search.best_params_)
print(f"Best tuned CV RMSE: {-search.best_score_:.4f}")

## 5) Holdout validation and diagnostics

Evaluate tuned model on validation set and inspect prediction behavior and residual distribution.

In [ ]:
X_valid = df_valid[feature_cols_for_final]
y_valid = df_valid[TARGET]

y_pred = best_model.predict(X_valid)

valid_rmse = rmse(y_valid, y_pred)
valid_mae = mean_absolute_error(y_valid, y_pred)
valid_r2 = r2_score(y_valid, y_pred)
valid_nasa = nasa_score(y_valid, y_pred)

print("Validation metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE:  {valid_mae:.4f}")
print(f"R2:   {valid_r2:.4f}")
print(f"NASA: {valid_nasa:.4f}")

residuals = y_valid - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

axes[0].scatter(y_valid, y_pred, alpha=0.25, color="#2E75B6")
line_min = min(float(y_valid.min()), float(y_pred.min()))
line_max = max(float(y_valid.max()), float(y_pred.max()))
axes[0].plot([line_min, line_max], [line_min, line_max], "r--")
axes[0].set_title("Predicted vs Actual")
axes[0].set_xlabel("Actual RUL clipped")
axes[0].set_ylabel("Predicted RUL clipped")

sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="#70AD47")
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual (actual - pred)")

axes[2].scatter(y_pred, residuals, alpha=0.25, color="#AB47BC")
axes[2].axhline(0, linestyle="--", color="black")
axes[2].set_title("Residuals vs Predicted")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.show()

# Checkpoint: prediction sanity
print("Prediction range:", (float(y_pred.min()), float(y_pred.max())))
print("Target range:", (float(y_valid.min()), float(y_valid.max())))

## 6) Feature importance and final checkpoint

Report feature effects for interpretability and capture reproducibility metadata.

In [ ]:
importance_df = None
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame(
        {
            "feature": feature_cols_for_final,
            "importance": best_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=importance_df.head(20),
        x="importance",
        y="feature",
        color="#1F4E79",
    )
    plt.title("Top 20 Feature Importances")
    plt.tight_layout()
    plt.show()

    print("Top 20 features:")
    print(importance_df.head(20))
else:
    print(f"Model '{best_model_name}' does not expose feature_importances_.")

model_checkpoint = {
    "best_baseline_model": str(best_model_name),
    "best_feature_space": str(best_feature_space),
    "n_features_used": int(len(feature_cols_for_final)),
    "valid_rmse": float(valid_rmse),
    "valid_mae": float(valid_mae),
    "valid_r2": float(valid_r2),
    "valid_nasa": float(valid_nasa),
}

print("Modeling checkpoint:")
print(model_checkpoint)